# Example: A Software Development Kit for the National Weather Service API
In this example, we build a small software development kit (SDK) for the application programming interface (API) of the [National Weather Service (NWS)](https://www.weather.gov/documentation/services-web-api). An SDK is a library of types and functions that makes the calls to an API for us. We use the SDK to download the hourly weather forecast for Ithaca, NY.

> __Learning Objectives:__
>
> By the end of this example, you should be able to:
>
> * __Call a RESTful API:__ A call to a RESTful API is an HTTP request to a URL that identifies a resource. We'll build the URL of the NWS points endpoint for Ithaca, NY and make an HTTP GET call from Julia.
> * __Process a structured API response:__ The NWS API returns its data as JSON text, a tree of nested objects and arrays. We'll parse the response into Julia dictionaries and read the forecast URL and the forecast periods from the tree.
> * __Apply the handler pattern:__ A handler is a function that processes the raw response of an API call before the response is returned to the caller. We'll pass a handler to the forecast call that returns the forecast as a table.

In addition to demonstrating how to call an API, the code that backs this example has some interesting design choices. Let's check them out!
___

## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course package, this example's code in the [src](src) folder, and the packages used here.

Let's set up our code environment:

In [1]:
# Load packages and paths from this notebook's local setup file -
include(joinpath(@__DIR__, "Include.jl"));

In addition to standard Julia libraries, we'll use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material.

The code for this example is in four files in the [src](src) folder:

* [src/Types.jl](src/Types.jl) holds the endpoint models, one type for each endpoint that we call.
* [src/Factory.jl](src/Factory.jl) holds the factory methods that construct the URL of a request.
* [src/Network.jl](src/Network.jl) holds the functions that make the HTTP calls.
* [src/Handler.jl](src/Handler.jl) holds the handlers, the functions that process a response.

### Constants
We'll set the latitude and longitude of our target location and the base URL of the NWS API. See the comments for additional details.

In [2]:
latitude = 42.443961; # latitude of Ithaca, NY (units: decimal degrees, north is positive)
longitude = -76.501881; # longitude of Ithaca, NY (units: decimal degrees, east is positive)
baseurl = "https://api.weather.gov"; # base URL of the National Weather Service API

___
## Task 1: Build a Grid Point Endpoint Model and Call the Points Endpoint
In this task, we build a model of the points endpoint and download the location data for Ithaca, NY.

The NWS computes its forecasts on a grid of 2.5 km squares. [The points endpoint](https://www.weather.gov/documentation/services-web-api#/default/point) returns information about a latitude and longitude pair, including the grid square that holds the location.

Let's build [a `MyWeatherGridPointEndpointModel` instance](src/Types.jl) by passing the `(latitude, longitude)` data as keyword arguments to the constructor:

In [3]:
model = MyWeatherGridPointEndpointModel(latitude = latitude, longitude = longitude)

MyWeatherGridPointEndpointModel(42.443961, -76.501881)

Next, we construct the Uniform Resource Locator (URL) of the endpoint using [a `build(...)` method](src/Factory.jl). The URL encodes how to reach the server (the `https` protocol and the host `api.weather.gov`) and the resource that we want (the path `/points/{latitude},{longitude}`).

We save the URL in the `points_url_string::String` variable:

In [4]:
points_url_string = build(baseurl, model)

"https://api.weather.gov/points/42.443961,-76.501881"

Now, let's make our first call! Notice the `syntax sugar`: we call the `MyWeatherGridPointEndpointModel` type as if it were a function.

> __What is happening behind the scenes?__ The [src/Network.jl](src/Network.jl) file defines a method that lets us call any subtype of `AbstractWeatherEndpointModel` with a URL string. The method does three things:
>
> * It makes an HTTP GET call to the URL with [the `HTTP.jl` package](https://juliaweb.github.io/HTTP.jl/stable/).
> * It checks the status code of the response, and throws an error if the code is not `200`.
> * It passes the body of the response to a handler. The default handler parses the JSON text into a dictionary.

We store the data returned from the API call in the `result_points` variable:

In [5]:
result_points = MyWeatherGridPointEndpointModel(points_url_string) # Hmmm. That's cool - a type as a function.

JSON.Object{String, Any} with 5 entries:
  "@context"   => Any["https://geojson.org/geojson-ld/geojson-context.jsonld", …
  "id"         => "https://api.weather.gov/points/42.444,-76.5019"
  "type"       => "Feature"
  "geometry"   => Object{String, Any}("type"=>"Point", "coordinates"=>Any[-76.5…
  "properties" => Object{String, Any}("@id"=>"https://api.weather.gov/points/42…

We can access the data in the dictionary using standard Julia dictionary syntax. For example, the `properties` field holds the data about our location:

In [6]:
result_points["properties"] # access the 'properties' field from the returned data

JSON.Object{String, Any} with 20 entries:
  "@id"                 => "https://api.weather.gov/points/42.444,-76.5019"
  "@type"               => "wx:Point"
  "cwa"                 => "BGM"
  "type"                => "land"
  "forecastOffice"      => "https://api.weather.gov/offices/BGM"
  "gridId"              => "BGM"
  "gridX"               => 44
  "gridY"               => 70
  "forecast"            => "https://api.weather.gov/gridpoints/BGM/44,70/foreca…
  "forecastHourly"      => "https://api.weather.gov/gridpoints/BGM/44,70/foreca…
  "forecastGridData"    => "https://api.weather.gov/gridpoints/BGM/44,70"
  "observationStations" => "https://api.weather.gov/gridpoints/BGM/44,70/statio…
  "relativeLocation"    => Object{String, Any}("type"=>"Feature", "geometry"=>O…
  "forecastZone"        => "https://api.weather.gov/zones/forecast/NYZ025"
  "county"              => "https://api.weather.gov/zones/county/NYC109"
  "fireWeatherZone"     => "https://api.weather.gov/zones/fire/NYZ210"
  

__What do we see?__ The `gridId`, `gridX`, and `gridY` fields identify the forecast office and the grid square for Ithaca, NY. The `forecast` and `forecastHourly` fields hold the URLs of the forecasts for that grid square.

___
## Task 2: Follow the Forecast Link and Download the Raw Forecast
In this task, we read the URL of the hourly forecast from the points response and call the forecast endpoint.

The NWS API returns the forecast URL directly from the points endpoint call. We extract the forecast URL and store it in the `forecast_url::String` variable.

> __Is this normal?__ No, this is unusual because the complete forecast URL is returned directly. Usually, we'd need to get some intermediate data from the endpoint call, and then build the next URL string ourselves. However, each API is unique in design.

Let's grab the URL string for the hourly forecast from the `result_points` dictionary:

In [7]:
forecast_url = result_points["properties"]["forecastHourly"]

"https://api.weather.gov/gridpoints/BGM/44,70/forecast/hourly"

Finally, we have the URL string for the weather forecast for Ithaca, NY. Let's call the NWS API again with the `forecast_url::String` variable, this time using [the `MyWeatherForecastEndpointModel` type](src/Types.jl), and look at what comes back:

In [8]:
result_forecast_raw = MyWeatherForecastEndpointModel(forecast_url) # notice the pattern - check out the code to see what is going on

JSON.Object{String, Any} with 4 entries:
  "@context"   => Any["https://geojson.org/geojson-ld/geojson-context.jsonld", …
  "type"       => "Feature"
  "geometry"   => Object{String, Any}("type"=>"Polygon", "coordinates"=>Any[Any…
  "properties" => Object{String, Any}("units"=>"us", "forecastGenerator"=>"Hour…

The forecast data is stored in the `properties` field, in a sub-field called `periods`. The `periods` field holds a list with one entry for each hour of the forecast.

Let's count the forecast periods, and then look at the first period:

In [9]:
periods = result_forecast_raw["properties"]["periods"]; # access the 'periods' field from the returned forecast data
println("Number of forecast periods: $(length(periods))")
periods[1] # the first forecast period

Number of forecast periods: 156


JSON.Object{String, Any} with 16 entries:
  "number"                     => 1
  "name"                       => ""
  "startTime"                  => "2026-11-12T15:00:00-05:00"
  "endTime"                    => "2026-11-12T16:00:00-05:00"
  "isDaytime"                  => true
  "temperature"                => 47
  "temperatureUnit"            => "F"
  "temperatureTrend"           => nothing
  "probabilityOfPrecipitation" => Object{String, Any}("unitCode"=>"wmoUnit:perc…
  "dewpoint"                   => Object{String, Any}("unitCode"=>"wmoUnit:degC…
  "relativeHumidity"           => Object{String, Any}("unitCode"=>"wmoUnit:perc…
  "windSpeed"                  => "9 mph"
  "windDirection"              => "N"
  "icon"                       => "https://api.weather.gov/icons/land/day/rain_…
  "shortForecast"              => "Rain Showers"
  "detailedForecast"           => ""

__What do we see?__ Each forecast period is a JSON object that holds fields such as `startTime`, `isDaytime`, `temperature`, and `windSpeed`. Some fields hold nested objects, e.g., the `probabilityOfPrecipitation` field. Thus, the response is a tree of data, which is common for RESTful APIs.

While this raw data is interesting, it is not convenient to work with. We can do better with a handler that processes the data before it is returned.

___
## Task 3: Process the Forecast with a Handler
In this task, we use a handler function to process the raw forecast data __before__ it is returned to the caller.

> __Why would we do this?__ We use a handler when we want the data from the API in a more convenient form, e.g., as [a `DataFrame` instance](https://dataframes.juliadata.org/stable/), or when we want to select or transform the data. Ultimately, this pattern makes our lives easier when we use the API data.

Notice the syntax sugar pattern again. We call the `MyWeatherForecastEndpointModel` type as a function with the `forecast_url::String` variable. We pass [the `process_forecast_response_dataframe(...)` function](src/Handler.jl) through the `handler` keyword argument:

In [10]:
result_forecast = MyWeatherForecastEndpointModel(forecast_url, handler = process_forecast_response_dataframe) # notice the handler pattern

Row,startTime,endTime,isDayTime,temperature,temperatureUnit,windSpeed,windDirection,shortForecast
,String,String,Bool,Int64,String,String,String,String
1,2026-11-12T15:00:00-05:00,2026-11-12T16:00:00-05:00,true,47,F,9 mph,N,Rain Showers
2,2026-11-12T16:00:00-05:00,2026-11-12T17:00:00-05:00,true,46,F,9 mph,N,Rain Showers
3,2026-11-12T17:00:00-05:00,2026-11-12T18:00:00-05:00,true,44,F,9 mph,N,Rain Showers
4,2026-11-12T18:00:00-05:00,2026-11-12T19:00:00-05:00,false,42,F,9 mph,N,Rain Showers
5,2026-11-12T19:00:00-05:00,2026-11-12T20:00:00-05:00,false,42,F,8 mph,N,Light Rain
6,2026-11-12T20:00:00-05:00,2026-11-12T21:00:00-05:00,false,41,F,8 mph,N,Light Rain
7,2026-11-12T21:00:00-05:00,2026-11-12T22:00:00-05:00,false,40,F,8 mph,N,Light Rain Likely
8,2026-11-12T22:00:00-05:00,2026-11-12T23:00:00-05:00,false,38,F,8 mph,N,Light Rain Likely
9,2026-11-12T23:00:00-05:00,2026-11-13T00:00:00-05:00,false,37,F,8 mph,N,Light Rain Likely


Now the data comes back as a `DataFrame`, which is much easier to work with! The handler kept eight fields from each forecast period and dropped the rest.

> __How does this work?__ The call passes the type, the URL, and the handler to a private function in [src/Network.jl](src/Network.jl). The private function makes the HTTP GET call and passes the body of the response to the handler. The call returns whatever the handler returns.

This approach is flexible. We can write a different handler for each way that we want to process the data. However, if we use code that someone else wrote, we need to know which handlers are available.

### Check: Forecast table
Let's check that the forecast link points back to the NWS API, and that the handler returned a table with the eight fields we expect:

In [11]:
@testset "NWS forecast table" begin
    expected_fields = ["startTime", "endTime", "isDayTime", "temperature", "temperatureUnit", "windSpeed", "windDirection", "shortForecast"];
    @test startswith(forecast_url, baseurl * "/") # the forecast link points back to the weather service
    @test result_forecast isa DataFrame # the handler returned a table
    @test nrow(result_forecast) > 0 # the table holds at least one forecast period
    @test names(result_forecast) == expected_fields # the columns are the fields the handler selects, in order
end;

Test Summary:      | Pass  

Total  Time
NWS forecast table |    4      4  0.3s


___
## Summary
In this example, we built an SDK for the National Weather Service API, called two of its endpoints, and returned the hourly forecast for Ithaca, NY as a table.

> __Key Takeaways:__
>
> * __Endpoints are often chained:__ We called the points endpoint with a latitude and longitude, and its response gave us the URL of the hourly forecast. We then called that URL to download the forecast.
> * __Raw responses are not always convenient:__ We parsed the JSON text of each response into nested dictionaries and lists. The forecast response held more fields than we needed, and some of them were nested.
> * __Handlers separate retrieval from processing:__ We passed a handler to the forecast call, and the call returned a table with the eight fields that the handler selects. The code that makes the HTTP call did not change.

We can now change the latitude and longitude to download the forecast for another location in the United States.
___